In this notebook, we will be:

- building a collage of images of plants that resembles an image of a pig
- optimize a noisy image so that the image is given a high classification score for a daisy by a VGG16 model
- introducing noise into an image of a goldfish so that the noise causes the image of the goldfish to be classified by a VGG16 model as a goldfinch bird.

Below we download our data, tiny ImageNet.

In [ ]:
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip -q tiny-imagenet-200.zip

Next, we import the libraries we'll be using.

In [ ]:
#from PIL import Image
!pip install keras=='2.3.1'
!pip install tensorflow=='2.2.0'

from keras.preprocessing import image
import numpy as np
from sklearn.model_selection import train_test_split

#pig = "n02395406"
# cats_0 = ['n01443537','n01629819','n01641577','n01644900','n01698640','n01742172',       'n01855672','n01882714','n02002724','n02056570','n02058221','n02074367',   'n02085620','n02094433','n02099601','n02099712','n02106662','n02113799',        'n02123045','n02123394','n02124075','n02125311','n02129165','n02132136',        'n02364673','n02395406','n02403003','n02410509','n02415577','n02423022',        'n02437312','n02480495','n02481823','n02486410','n02504458','n02509815']
# cats_1 = ['n01770393','n01774384','n01774750','n01784675','n02165456','n02190166',          'n02206856','n02226429','n02231487','n02233338','n02236044','n02268443', 'n02279972','n02281406']

Here, we load images from the categories we need, `pigs` and `plants` as grayscale images.

In [ ]:
c = "n02395406"
pigs = []
for i in range(500):
  img = image.load_img("tiny-imagenet-200/train/"+c+"/images/"+c+"_"+str(i)+".JPEG", color_mode="grayscale")
  img_arr = image.img_to_array(img)
  img_arr = img_arr.flatten()
  img_arr = img_arr.reshape(64,64,1)
  pigs += [img_arr]
c = "n07715103" # cauliflower
#c = "n07734744" # mushroom
plants = []
#for c in ["n07715103","n07734744", "n07753592","n12267677","n07768694", "n07749582", "n07747607", "n07711569", "n07583066"]: # and banana
if True:
  for i in range(500):
    img = image.load_img("tiny-imagenet-200/train/"+c+"/images/"+c+"_"+str(i)+".JPEG", color_mode="grayscale")
    img_arr = image.img_to_array(img)
    img_arr = img_arr.flatten()
    img_arr = img_arr.reshape(64,64,1)
    plants += [img_arr]

We define two functions:

- `sim` which calculates our similartity metric
- `mostsim` which finds the image most similar to the `target` image among the `sources` image 

In [ ]:
def sim(a,b):
#  print(a.shape, b.shape)
  return(-np.sum((a-b)**2))

def mostsim(sources, target):
  bestsim, bestsource = sim(sources[0], target), sources[0]
  for source in sources:
    simsource = sim(source, target)
    if simsource > bestsim: bestsim, bestsource = simsource, source
  return(bestsource)


This `make64` function scales up the `arr` image by `factor`

In [ ]:
def make64(arr,factor):
  out = np.empty((64,64,1))
  for x in range(out.shape[0]):
    for y in range(out.shape[1]):
      out[x,y] = arr[x//factor,y//factor]
  return(out)

Here's our target pig.

In [ ]:
!wget https://github.com/mlittmancs/great_courses_ml/raw/master/imgs/pig-bw.jpg

Load in the pig.

In [ ]:
img = image.img_to_array(image.load_img("pig-bw.jpg", target_size=(64,64),color_mode="grayscale"))

We use the functions we described above to build a large image out of our `plants` images that resembles the `pig` image.

In [ ]:
# pig = pigs[217]

pig = img

factor = 32
subsize = 64//factor
frankenpig = np.empty((64*factor,64*factor,1))
display(image.array_to_img(pig))
for x in range(factor):
  for y in range(factor):
    subpig = pig[(x*subsize):(x*subsize+subsize),(y*subsize):(y*subsize+subsize),]
    bigsubpig = make64(subpig,factor)
    plant = mostsim(plants,bigsubpig)
    frankenpig[(x*64):(x*64+64),(y*64):(y*64+64)] = plant #bigsubpig # mostsim(plants, bigsubpig)

frankenpigimg = image.array_to_img(frankenpig)
frankenpigimg.resize((512,512))

Next we import additional libraries and get our VGG16 model which we will be using to make adversarial images that fools our VGG16 model.

In [ ]:
import numpy as np
from keras.applications import vgg16
from keras import backend as K

 
# build the VGG16 network with ImageNet weights
model = vgg16.VGG16(weights='imagenet', include_top=True)

We import additional keras functions here.

In [ ]:
from keras.preprocessing import image
from keras.applications.vgg16 import preprocess_input, decode_predictions

The `rand_image` function builds a random noise image.  We will display an example image, `img`, below.

In [ ]:
def rand_image():
  img_arr = np.random.rand(1,224,224,3)*128+128
  img_arr = preprocess_input(img_arr)
  return(img_arr)
 
img = rand_image()
display(image.array_to_img(img[0]))

We print the top 10 predicted classes by our `model` for our example random image, `img`.

In [ ]:
preds = model.predict(img)
 
print('Predicted:', decode_predictions(preds, top=10)[0])


Next we'll create a `daisy_score` which is the prediction given by  our VGG16 model for the daisy class.  We print out the `daisy_score` for our random image, `img`.

In [ ]:
def daisy_score(img):
  preds = model.predict(img)
  return(preds[0][985])
 
daisy_score(img)

In this code block we: 
- generate 1000 random images
- get the `daisy_score` prediction for each random image
- store the `img` with the highest `daisy_scores` compared to all other proposed images as `bestimg` 
- print the highest `daisy_score`

In [ ]:
bestimg = None
maxdaisy = 0.0
for i in range(1000):
  img = rand_image()
  score = daisy_score(img)
  if score > maxdaisy: maxdaisy, bestimg = score, img

daisy_score(bestimg)

We `display` the imaage with the higest `daisy_score`, `bestimg`

In [ ]:
display(image.array_to_img(bestimg[0]))

We collect the layer information of our `model` in `layer_dict` build an `objective` relative to the predictions for the daisy `target_class`, similar to our `daisy_score` function.  We will use this objective to generate an image that can lead the model to give the image a high prediction value for the daisy class.

In [ ]:
# get the symbolic outputs of each "key" layer (we gave them unique names).
layer_dict = dict([(layer.name, layer) for layer in model.layers[1:]])
  
target_class = 985
 
input_tensor = model.input
x = layer_dict['predictions'].output[:, target_class]
objective = K.sum(x)

We calculate the gradient, `grads`, with respect to the `input_tensor` image and the daisy `objective`. We then define an `iterate` function in terms of the `input_tensor`, `objetive` and `grads`.

In [ ]:
grads = K.gradients(objective, input_tensor)[0]
grads /= (K.sqrt(K.mean(K.square(grads))) + 1e-12)
iterate = K.function([input_tensor], [objective, grads])

In this code block, we will iteratively add to the random image, `img`, based on our daisy `objective` until is it greater then or equal to 0.9999999.

In [ ]:
obj_value = 0.0
while obj_value < 0.9999999: 
 obj_value, grads_value = iterate([img])
 print(obj_value)
 img += grads_value

Below is our image optimized for the daisy class.

In [ ]:
display(image.array_to_img(img[0]))

In the following code blocks, we will be adding noise to a golsfish image so that VGG16 outputs a high prediction value for the goldfinch class. 

Below we load the VGG16 `model`.

In [ ]:
import numpy as np
from keras.applications import vgg16
from keras import backend as K
 
# build the VGG16 network with ImageNet weights
model = vgg16.VGG16(weights='imagenet', include_top=True)
print('Model loaded.')

We download an image of a goldfish.

In [ ]:
# !wget http://c2.peakpx.com/wallpaper/137/72/865/veiltail-fish-goldfish-swim-aquarium-wallpaper-preview.jpg
!wget https://github.com/mlittmancs/great_courses_ml/raw/master/imgs/goldfish.jpg

We load and display the goldfish image, `img_arr`, below.

In [ ]:
from keras.preprocessing import image
from keras.applications.vgg16 import preprocess_input, decode_predictions

img = image.load_img("goldfish.jpg")
img = img.resize((224,224))
img_arr = image.img_to_array(img)
img_arr = img_arr.flatten()
img_arr = img_arr.reshape(1,224,224,3)
display(image.array_to_img(img_arr[0]))

Next we print the top 3 predicitions for this goldfish image by VGG16.

In [ ]:
preds = model.predict(img_arr)

print('Predicted:', decode_predictions(preds, top=3)[0])

In this code block we:

- collect the layer information for each layer in the `model` using `layer_dict`
- identify goldfinches as our `target_class` and define the predicitions for `target_class` as our `objective`
- calcuate the gradient, `grads` with respect to the `input_tensor` image and `obective`
- call the `iterate` function to add noise to the `input_img_data` image of the goldfish until the predtion for the goldfinch class is greater or equal to 0.8, printing the prediction for the image belonging to the goldfinch class each time we make an update
- `display` the new version of `input_img_data`

In [ ]:
# get the symbolic outputs of each "key" layer (we gave them unique names).
layer_dict = dict([(layer.name, layer) for layer in model.layers[1:]])
  
target_class = 11 # 9 is ostrich, 1 is goldfish, 11 is goldfinch

learning_rate = 0.1

input_tensor = model.input
x = layer_dict['predictions'].output[:, target_class]
objective = K.sum(x)
grads = K.gradients(objective, input_tensor)[0]
grads /= (K.sqrt(K.mean(K.square(grads))) + 1e-12)
iterate = K.function([input_tensor], [objective, grads])
input_img_data = img_arr

loss_value = 0.0
while loss_value < 0.8: 
  loss_value, grads_value = iterate([input_img_data])
  print(loss_value)
  input_img_data += grads_value

display(image.array_to_img(input_img_data[0]))

We display the top 3 predictions for the new image.

In [ ]:
preds = model.predict(input_img_data)

print('Predicted:', decode_predictions(preds, top=3)[0])